In [2]:
import json
import sqlite3


def fetch_notegroup_json(db_path: str, table_name: str, notegroup_id: int) -> str:
    """
    Fetch record(s) for a given notegroupID from `table_name` and return
    them as a JSON string.

    Format:
    - For "notegroups": notegroupID IS the primary key, so there's only
      one record. Returned as a flat dict: {"date": ..., "data_source_category": ...}
      (NOT wrapped in a list, and NOT nested under the table name).
    - For all other tables ("participants", "questions", "answers"):
      returned as {"<table_name>": [record, record, ...]}, one entry per
      row belonging to that notegroup.
    """
    records = _fetch_records_by_notegroup(db_path, table_name, notegroup_id)

    if table_name == "notegroups":
        record = records[0] if records else {}
        record = {k: v for k, v in record.items() if k in ("date", "data_source_category")}
        json_str = json.dumps(record, ensure_ascii=False)
        return json_str

    processed = []
    for record in records:
        if table_name == "participants":
            record.pop("notegroupID", None)
            record.pop("country_staying_in", None)
            record.pop("labor_market_region", None)

            projectID_age = record.pop("projectID_age")
            age = next(iter(projectID_age.values()))

            # rebuild dict so "age" lands right after "gender"
            reordered = {}
            for key, value in record.items():
                reordered[key] = value
                if key == "gender":
                    reordered["age"] = age
            record = reordered

        elif table_name == "questions":
            record.pop("notegroupID", None)
            record.pop("question_type", None)

        elif table_name == "answers":
            record.pop("notegroupID", None)
            record.pop("projectID", None)
            record.pop("answer_extraction_LLM", None)
            record.pop("sentiment_score_LLM", None)

        processed.append(record)

    result = {table_name: processed}
    json_str = json.dumps(result, ensure_ascii=False)
    return json_str


def _fetch_records_by_notegroup(db_path: str, table: str, notegroup_id: int) -> list[dict]:
    """Fetch all rows for a given notegroupID from `table`, decoding JSONB columns."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.execute(
            f"SELECT * FROM {table} WHERE notegroupID = ?", (notegroup_id,)
        )
        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()

    records = []
    for row in rows:
        record = {}
        for col, val in zip(columns, row):
            if isinstance(val, str):
                try:
                    val = json.loads(val)  # restore JSONB columns
                except (json.JSONDecodeError, ValueError):
                    pass
            record[col] = val
        records.append(record)
    return records

In [3]:
from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor
from oral_notes.s2_transform.participant_llm_reducer import ParticipantReducer
from oral_notes.prompt_combiner_v3 import PromptCombiner
from openai import OpenAI
from config.config import OPENAI_API_KEY
def refine_notegroup(notegroup_id: int, llm_model: str = "gpt-5.1", max_rounds: int = 3) -> str:
    """
    Refine the previously-extracted 1recordT ("notegroups") output for a
    given notegroupID: re-loads the source transcript(s) and file path,
    fetches the existing DB result, and asks the refiner LLM to either
    "pass" it or return a corrected JSON object.
    """
    task = "1recordT"
    # ── 1. Look up source URLs for this notegroup ───────────────────────────
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT project_name, phase, note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()
        if row is None:
            raise ValueError(f"No notegroup found with ID {notegroup_id}")
        project_name, phase, note_url_qa, note_url_participant = row

    # ── 2. Re-load and extract text + file path(s), same as ETL_baseline ───
    file_loader = GoogleDriveLoader(service_account_file)
    extractor = TextExtractor()

    all_texts = {}
    all_drive_paths = {}
    for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
        if not url:
            continue
        result = file_loader.load(url)
        text = extractor.extract(result)
        all_texts[label] = f"[Data source: {result['name']}]\n{text}"
        all_drive_paths[label] = result['drive_path']
    combined_drive_paths = "|".join(all_drive_paths.values())

    # ── 3. Run participant reduction if applicable, same as ETL_baseline ───
    has_participant = "PARTICIPANT" in all_texts
    if has_participant:
        reducer = ParticipantReducer(
            pipeline_type=pipeline_type,
            prompt_path_ParReducer=prompt_path_ParReducer,
            all_texts=all_texts,
            notegroup_id=notegroup_id,
        )
        combined_text = reducer.build_combined_text()
    else:
        combined_text = all_texts["QA"]

    text_doc = combined_text
    file_path_doc = combined_drive_paths

    # ── 4. Fetch the initial DB result for this notegroup ───────────────────
    current_result = fetch_notegroup_json(DB_PATH, "notegroups", notegroup_id)
    print(f"--- Initial DB result for notegroupID={notegroup_id} ---")
    print(current_result)

    # ── 5. Set up prompt combiner + LLM client (reused across rounds) ───────
    combiner = PromptCombiner(schema_path=schema_path)
    system_prompt = combiner.load_prompts_system(prompt_path_refiner_1recordT)
    pass_placeholder = json.loads(combiner.build_pass_placeholder(task))

    client = OpenAI(
        api_key=OPENAI_API_KEY,
        base_url="https://llmproxy.uva.nl/v1",
        timeout=500.0
    )

    # ── 6. Refinement loop ────────────────────────────────────────────────────
    for round_num in range(1, max_rounds + 1):
        user_prompt = combiner.build_prompt_user_refiner_1recordT(
            prompt_path=prompt_path_refiner_1recordT,
            file_path_doc=file_path_doc,
            text_doc=text_doc,
            json_result_lastcall=current_result
        )

        response = client.chat.completions.create(
            model=llm_model,
            temperature=0,
            response_format=combiner.to_json_schema(task),
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        result = response.choices[0].message.content.strip()
        parsed = json.loads(result)
        print(f"--- Round {round_num}/{max_rounds} result for notegroupID={notegroup_id} ---")
        print(result)

        if parsed == pass_placeholder:
            print(f"--- Passed on round {round_num}, returning previous result ---")
            return current_result

        current_result = result

    print(f"--- Reached max_rounds={max_rounds} without a pass, returning last result ---")
    return current_result

In [4]:
DB_PATH = "DB/oedb_baseline_v3.db"
schema_path = "data/metadata_DB/schema_v3.yaml"
prompt_path_refiner_1recordT = "data/prompt_templates/refiner/prompt_refiner_1recordT.yaml"
prompt_path_ParReducer = "data/prompt_templates/prompt_ParReducer.yaml"
service_account_file = "config/service_account_key.json"
pipeline_type = "refiner_test"

result = refine_notegroup(notegroup_id=1)
print(result)

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1


2026-07-01 22:30:15 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 6465 (cached: 0), out: 1175, cost: $0.019831
2026-07-01 22:30:15 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 1) ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | F.A |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | A.A |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | Y.A |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | L.H |
| Ahmad Noman | Male | Yemen | B1 | 24 | Helmond | A.N |
| Zaid Kurami | Male | Yemen | B1 | 25 | Hemlond | Z.K |
[/TABLE]


--- Initial DB result for notegroupID=1 ---
{"date": null, "data_source_category": "expertpool"}
--- Round 1/3 result for notegroupID=1 ---
{"date": "pass", "data_source_category": "pass"}
--- Passed on round 1, returning previous result ---
{"date": null, "data_source_category": "expertpool"}


In [5]:
result = refine_notegroup(notegroup_id=17)
print(result)

Loading: Interview 3.18 BOOST (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1133 - AMIF Amsterdam/06 Diepteinterviews /Interviews BOOST/Interview 3.18 BOOST
--- Initial DB result for notegroupID=17 ---
{"date": null, "data_source_category": "diepteinterview"}
--- Round 1/3 result for notegroupID=17 ---
{"date": "pass", "data_source_category": "pass"}
--- Passed on round 1, returning previous result ---
{"date": null, "data_source_category": "diepteinterview"}


In [6]:
result = refine_notegroup(notegroup_id=21)
print(result)

Loading: NOTITIES_Floris_en_Anne.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_Floris_en_Anne.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)


2026-07-01 22:33:00 | INFO     | utils.token_logger | Token usage [ParticipantReducer] attempt 1 — in: 22186 (cached: 0), out: 723, cost: $0.034963
2026-07-01 22:33:00 | INFO     | oral_notes.s2_transform.participant_llm_reducer | === Result: reduced participants (from 21) ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتيت

--- Initial DB result for notegroupID=21 ---
{"date": null, "data_source_category": "focus group"}
--- Round 1/3 result for notegroupID=21 ---
{"date": "pass", "data_source_category": "pass"}
--- Passed on round 1, returning previous result ---
{"date": null, "data_source_category": "focus group"}


In [15]:
from oral_notes.prompt_combiner_v3 import PromptCombiner

schema_path = "data/metadata_DB/schema.yaml"
combiner = PromptCombiner(schema_path=schema_path)

result = combiner.build_pass_placeholder("answers")
print(result)

{"answers": [{"answerID": "pass", "questionID": "pass", "participantID": "pass", "answer_content_oriLAN": "pass", "answer_content_EN": "pass"}]}


In [11]:
db_path = "DB/oedb_baseline_v3.db"
notegroup_id = 1

for table in ["notegroups", "participants", "questions", "answers"]:
    result = fetch_notegroup_json(db_path, table, notegroup_id)
    print(f"--- {table} ---")
    print(result)
    print()

--- notegroups ---
{"date": null, "data_source_category": "expertpool"}

--- participants ---
{"participants": [{"participantID": 1, "full_name": "Ahmad Ahmad", "session_identifier": "A.A", "gender": "Male", "age": 35, "learning_route": "B1", "participant_group": null, "status": null, "place_of_origin": "Syria", "language_group": null, "first_arrival_date": null, "municipality": "Helmond", "other_information": {}}, {"participantID": 2, "full_name": "Yasmine Ahmad", "session_identifier": "Y.A", "gender": "Female", "age": 40, "learning_route": "B1", "participant_group": null, "status": null, "place_of_origin": "Syria", "language_group": null, "first_arrival_date": null, "municipality": "Helmond", "other_information": {}}, {"participantID": 3, "full_name": "Layla Hamliko", "session_identifier": "L.H", "gender": "Female", "age": 50, "learning_route": "B1", "participant_group": null, "status": null, "place_of_origin": "Syria", "language_group": null, "first_arrival_date": null, "municipalit